<a href="https://colab.research.google.com/github/BrionyMeng/Colab-Temp/blob/HEST-alignment/part1_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7 — Assignment, Part 1: Multi-modal Alignment

**DD2361 / FDD3020 — Deep Learning Methods for Biomedical Image Analysis**

---

In the [Multi-modal Alignment](https://example.invalid) lecture we asked: given an H&E
image patch and a spatial-transcriptomics (ST) expression profile, how does a model learn
*which patch goes with which spot*, when we have no clean index to look it up by?

In this notebook you will build that model. It follows the recipe behind
**BLEEP** (Xie et al., 2023): two encoders, one contrastive loss, and an evaluation by
**retrieval** — given a patch we have never seen, can the model find its true expression
profile in a large bank of candidates?

### What you will do

| | |
|---|---|
| **TODO 1** | Implement the **InfoNCE** contrastive loss |
| **TODO 2** | Implement **top-k retrieval accuracy** |
| **TODO 3** | Implement a **negative-free** loss, and watch it collapse |
| Experiments | Temperature sweep · number of negatives · collapse diagnostic |

### Ground rules

* You only write code inside the cells marked `### YOUR CODE HERE`. Everything else is
  given — read it, but you do not need to change it.
* Each function you write is followed by a **self-test cell**. Run it. If it does not
  print `ALL TESTS PASSED`, fix your function *before* running the experiments — the
  experiments will refuse to start otherwise.
* Everything is seeded. If you do not change the given code, your numbers should match
  your classmates' to within a small margin.
* **Expected runtime: about 3–8 minutes of compute in total.** If something is running for
  much longer than that, stop and check your implementation.

### What to hand in

Run the whole notebook top to bottom, then submit:

1. this notebook **with all outputs visible**, and
2. the two files the final cell writes: `part1_results.csv` and `part1_figure.png`.

Answer the short written questions at the bottom in the markdown cells provided.

## 0. Setup

*(Given — just run it.)*

In [1]:
import os
import io
import json
import time
import math
import random
import warnings
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE)


def set_seed(seed: int = 0):
    """Seed every source of randomness we use, so results are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

# ---- plotting style (a small, colour-blind-safe palette used throughout) -------------
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
C_INK, C_MUTED, C_GRID = "#0b0b0b", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": C_INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "text.color": C_INK, "grid.color": C_GRID, "grid.linewidth": 0.8,
    "font.size": 10, "figure.dpi": 110,
})

PyTorch 2.11.0+cpu | device: cpu


### Get the data

We have pre-extracted the features for you, so that this notebook runs in minutes rather
than days. Specifically:

* **Image side** — each ST spot's surrounding 224×224 H&E patch (20× magnification) was
  passed through a frozen, pretrained encoder, giving a 512-dimensional vector.
* **Expression side** — the spot's raw transcript counts were normalised and
  log-transformed, and restricted to the 256 most variable genes.
* We also kept a small set of 64×64 thumbnails so you can actually *look* at the tissue.

You are therefore working with a real biomedical alignment problem — you are simply not
paying for the encoder forward passes.

> **Note on the split.** The train / val / test split is **by slide**, never by spot.
> Two spots from the same slide share a patient, a staining batch and a sequencing run;
> if they were split at random, a model could score well by recognising the batch rather
> than the biology. This is the "batch effect" pitfall from the Summary lesson, and it is
> the single most common way to fool yourself with this kind of data.

In [2]:
DATA_URL = "PUT_THE_COURSE_DOWNLOAD_URL_HERE"   # e.g. a Canvas / Zenodo direct link
DATA_FILE = "hest_alignment.npz"

if not os.path.exists(DATA_FILE):
    print("downloading ...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

d = np.load(DATA_FILE, allow_pickle=True)

IMG_FEAT = d["img_feat"]        # (N, 512) float32 — frozen H&E patch embeddings
EXPR = d["expr"]                # (N, 256) float32 — log-normalised expression
SPLIT = d["split"]              # (N,)     str     — "train" / "val" / "test"
SAMPLE_ID = d["sample_id"]      # (N,)     int     — which slide the spot came from
THUMBS = d["thumbs"]            # (M, 64, 64, 3) uint8 — thumbnails for test spots
THUMB_IDX = d["thumb_idx"]      # (M,)     int     — index into the arrays above

print(f"{len(IMG_FEAT):,} spots from {len(np.unique(SAMPLE_ID))} slides")
for sp in ["train", "val", "test"]:
    m = SPLIT == sp
    print(f"  {sp:5s}  {m.sum():6,d} spots   {len(np.unique(SAMPLE_ID[m])):3d} slides")
print(f"  image features {IMG_FEAT.shape[1]}-d | expression {EXPR.shape[1]}-d")

downloading ...


ValueError: unknown url type: 'PUT_THE_COURSE_DOWNLOAD_URL_HERE'

### Look at the two modalities

Before modelling anything, look at what you are actually aligning. On the left, tissue.
On the right, the same location's expression profile. They describe the same
100-micrometre patch of tumour, and they look nothing like each other — that heterogeneity
is the whole problem.

In [ ]:
set_seed(0)
_show = np.random.choice(len(THUMBS), 4, replace=False)
fig, axes = plt.subplots(4, 2, figsize=(9, 8), gridspec_kw={"width_ratios": [1, 3.2]})
for row, j in enumerate(_show):
    axes[row, 0].imshow(THUMBS[j])
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    axes[row, 0].set_ylabel(f"spot {THUMB_IDX[j]}", fontsize=8, color=C_MUTED)
    axes[row, 1].bar(np.arange(EXPR.shape[1]), EXPR[THUMB_IDX[j]],
                     color=C_AQUA, width=1.0, linewidth=0)
    axes[row, 1].set_ylabel("log1p expr", fontsize=8)
    axes[row, 1].grid(axis="y", alpha=0.6)
    axes[row, 1].set_axisbelow(True)
    if row < 3:
        axes[row, 1].set_xticklabels([])
axes[0, 0].set_title("H&E patch", fontsize=10, loc="left")
axes[0, 1].set_title("matched expression profile (256 genes)", fontsize=10, loc="left")
axes[3, 1].set_xlabel("gene index")
plt.tight_layout()
plt.show()

frac_zero = (EXPR == 0).mean()
print(f"fraction of zero entries in the expression matrix: {frac_zero:.1%}")
print("(sparsity like this is normal for spatial transcriptomics — most genes are not")
print(" detected at most spots, which is part of why the negatives are ambiguous)")

## 1. The model

*(Given.)*

Two **projection heads**, one per modality. Each maps its input into a shared
128-dimensional space and L2-normalises the output, so that a dot product between two
embeddings *is* their cosine similarity. This is the "coordinated representation" from the
lecture: separate encoders, one common space.

We standardise both modalities using **training-set statistics only**.

In [ ]:
def get_split(name):
    m = SPLIT == name
    return (torch.tensor(IMG_FEAT[m]).float(), torch.tensor(EXPR[m]).float(),
            np.where(m)[0])


X_tr, E_tr, IDX_tr = get_split("train")
X_va, E_va, IDX_va = get_split("val")
X_te, E_te, IDX_te = get_split("test")

_mx, _sx = X_tr.mean(0), X_tr.std(0) + 1e-6
_me, _se = E_tr.mean(0), E_tr.std(0) + 1e-6
X_tr, X_va, X_te = [(t - _mx) / _sx for t in (X_tr, X_va, X_te)]
E_tr, E_va, E_te = [(t - _me) / _se for t in (E_tr, E_va, E_te)]

D_IMG, D_EXPR, D_EMBED = X_tr.shape[1], E_tr.shape[1], 128


class ProjectionHead(nn.Module):
    """Small MLP that maps one modality into the shared, L2-normalised space."""

    def __init__(self, d_in, d_out=D_EMBED, d_hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


print(f"image head: {D_IMG} -> {D_EMBED} | expression head: {D_EXPR} -> {D_EMBED}")

## TODO 1 — the InfoNCE loss

This is the heart of the assignment. Recall the loss from the lecture:

$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log
\frac{\exp\big(\mathrm{sim}(z_A^i, z_B^i)/\tau\big)}
     {\sum_{j=1}^{N}\exp\big(\mathrm{sim}(z_A^i, z_B^j)/\tau\big)}
$$

Read that as an $N$-way classification problem: *"for patch $i$, which of the $N$ spots in
this batch is the true partner?"* The correct answer for row $i$ is always column $i$ —
which means you can implement this with a plain cross-entropy against
`targets = [0, 1, 2, ..., N-1]`, and you do **not** need to write a softmax yourself.

Two details that matter:

* The inputs are already L2-normalised, so `z_a @ z_b.T` is the full matrix of cosine
  similarities. Divide it by `temperature` to get the logits.
* Make it **symmetric**: the formula above only queries patches against spots. Do the same
  in the other direction (spots against patches) and average the two. This is what CLIP
  does, and it usually trains more stably.

In [ ]:
def info_nce_loss(z_a: torch.Tensor, z_b: torch.Tensor, temperature: float) -> torch.Tensor:
    """Symmetric InfoNCE / CLIP contrastive loss.

    Parameters
    ----------
    z_a : (B, D) tensor, already L2-normalised — e.g. image embeddings.
    z_b : (B, D) tensor, already L2-normalised — e.g. expression embeddings.
          Row i of `z_b` is the TRUE partner of row i of `z_a`.
    temperature : float, the tau in the formula above.

    Returns
    -------
    A scalar tensor (0-dimensional) that can be back-propagated through.
    """
    ### YOUR CODE HERE ###
    # 1. Build the (B, B) matrix of similarities between every z_a and every z_b,
    #    and divide it by `temperature`. These are your logits.
    # 2. The correct class for row i is i. Build that target vector.
    #    (Careful: it must live on the same device as z_a.)
    # 3. Cross-entropy in the a->b direction (rows are queries).
    # 4. Cross-entropy in the b->a direction (columns are queries: transpose the logits).
    # 5. Return the mean of the two.
    raise NotImplementedError("Implement info_nce_loss")

### Self-test for TODO 1

*(Given — run it and make sure it passes.)*

The tests below are worth reading, because they tell you what the loss *means*:

* With **random** embeddings and a large temperature, the model has no information, so it
  guesses uniformly over the $B$ candidates and the loss must be $\ln B$.
* With **perfectly aligned** embeddings the true pair wins by a mile, so the loss is ~0.
* Swapping the roles of the two modalities must not change the answer (that is what
  "symmetric" means).

In [ ]:
def _check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    return ok


def run_selftest(test_fn, name):
    """Run a self-test, but give a friendly message if the function is still a stub."""
    try:
        return test_fn()
    except NotImplementedError:
        print(f"  {name} has not been implemented yet.")
        print("  Fill in the cell above, then re-run this cell.\n")
        return False


def test_info_nce():
    print("Testing info_nce_loss ...")
    g = torch.Generator().manual_seed(1)
    ok = True

    B, D = 64, 512
    a = F.normalize(torch.randn(B, D, generator=g), dim=-1)
    b = F.normalize(torch.randn(B, D, generator=g), dim=-1)

    # 1. returns a scalar
    out = info_nce_loss(a, b, 1.0)
    ok &= _check("returns a 0-dim scalar tensor", torch.is_tensor(out) and out.dim() == 0,
                 f"got {type(out).__name__} with shape {tuple(out.shape) if torch.is_tensor(out) else '-'}")

    # 2. uninformative embeddings -> ln(B)
    expected = math.log(B)
    ok &= _check("random embeddings give ~ln(B)", abs(float(out) - expected) < 0.15,
                 f"got {float(out):.3f}, expected ~{expected:.3f}")

    # 3. perfect alignment -> ~0
    perfect = info_nce_loss(a, a.clone(), 0.07)
    ok &= _check("perfectly aligned pairs give ~0", float(perfect) < 0.05,
                 f"got {float(perfect):.4f}")

    # 4. symmetry
    l1, l2 = info_nce_loss(a, b, 0.07), info_nce_loss(b, a, 0.07)
    ok &= _check("symmetric in its two arguments", abs(float(l1) - float(l2)) < 1e-4,
                 f"{float(l1):.5f} vs {float(l2):.5f}")

    # 5. mismatching the pairs must hurt
    shuffled = a[torch.randperm(B, generator=g)]
    ok &= _check("wrong pairing scores worse than right pairing",
                 float(info_nce_loss(a, shuffled, 0.07)) > float(perfect) + 1.0)

    # 6. differentiable
    p = torch.nn.Parameter(torch.randn(B, D, generator=g))
    info_nce_loss(F.normalize(p, dim=-1), b, 0.07).backward()
    ok &= _check("gradient flows to the inputs",
                 p.grad is not None and torch.isfinite(p.grad).all())

    print("ALL TESTS PASSED\n" if ok else "SOME TESTS FAILED — fix the function above.\n")
    return ok


TEST1_OK = run_selftest(test_info_nce, "info_nce_loss")

## TODO 2 — retrieval accuracy

A contrastive loss going down is not evidence of anything by itself. The question we
actually care about is the one BLEEP asks: **given a patch we have never seen, can we find
its true expression profile among a large bank of candidates?**

Concretely: build the similarity matrix between $N$ query embeddings and $N$ gallery
embeddings, where gallery item $i$ is the true match for query $i$. For each query, the
**rank** of the true match is

$$\text{rank}_i = 1 + \#\{\, j : \text{sim}(q_i, g_j) > \text{sim}(q_i, g_i) \,\}$$

i.e. one plus the number of *wrong* candidates that scored **strictly higher** than the
right one. Top-$k$ accuracy is then the fraction of queries with $\text{rank}_i \le k$.

> Use the strict `>` above, not `>=`. With ties (which happen if a model collapses) that
> choice changes the numbers a lot, and we want everyone's answer to agree.

Also return the **median rank**, which is more informative than top-1 when accuracy is
low: "the right answer is typically 9th out of 2000" is a much clearer statement than
"top-1 accuracy is 18%".

In [ ]:
def topk_retrieval_accuracy(z_query: torch.Tensor, z_gallery: torch.Tensor,
                            ks=(1, 5, 10)) -> dict:
    """Retrieval metrics for a set of query/gallery embeddings.

    Parameters
    ----------
    z_query   : (N, D) tensor, L2-normalised.
    z_gallery : (N, D) tensor, L2-normalised. Row i is the TRUE match for query i.
    ks        : which top-k accuracies to compute.

    Returns
    -------
    dict with a float entry "top{k}" for every k in `ks`, plus "median_rank".
    Example: {"top1": 0.18, "top5": 0.40, "top10": 0.53, "median_rank": 9.0}
    """
    ### YOUR CODE HERE ###
    # 1. Similarity matrix between queries and gallery: shape (N, N).
    # 2. For every query i, pull out the similarity of its TRUE match — that is the
    #    diagonal of the matrix.
    # 3. Count, per row, how many entries are STRICTLY greater than that diagonal value,
    #    and add 1. That is the rank of the true match.
    # 4. top-k accuracy = fraction of ranks <= k. Also compute the median rank.
    #    Return plain Python floats (use float(...)), not tensors.
    raise NotImplementedError("Implement topk_retrieval_accuracy")

### Self-test for TODO 2

*(Given.)* Test 2 is a tiny hand-checkable case — work through it on paper if the test
fails, it will tell you immediately whether your rank convention is off by one.

In [ ]:
def test_retrieval():
    print("Testing topk_retrieval_accuracy ...")
    ok = True

    # 1. identity: every query IS its gallery item, so it must be rank 1
    z = F.normalize(torch.randn(50, 32, generator=torch.Generator().manual_seed(2)), dim=-1)
    r = topk_retrieval_accuracy(z, z.clone())
    ok &= _check("perfect case gives top1 = 1.0", abs(r["top1"] - 1.0) < 1e-9, f"got {r['top1']}")
    ok &= _check("perfect case gives median rank 1", abs(r["median_rank"] - 1.0) < 1e-9,
                 f"got {r['median_rank']}")

    # 2. hand-computable 2-query case where the true match is ALWAYS second
    q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
    g = torch.tensor([[0.6, 0.8], [0.8, 0.6]])       # sim = [[.6,.8],[.8,.6]]
    r = topk_retrieval_accuracy(q, g, ks=(1, 2))     # true match is rank 2 for both
    ok &= _check("hand case: top1 = 0.0", abs(r["top1"]) < 1e-9, f"got {r['top1']}")
    ok &= _check("hand case: top2 = 1.0", abs(r["top2"] - 1.0) < 1e-9, f"got {r['top2']}")
    ok &= _check("hand case: median rank = 2", abs(r["median_rank"] - 2.0) < 1e-9,
                 f"got {r['median_rank']}")

    # 3. unrelated embeddings -> chance level
    g2 = torch.Generator().manual_seed(3)
    a = F.normalize(torch.randn(500, 64, generator=g2), dim=-1)
    b = F.normalize(torch.randn(500, 64, generator=g2), dim=-1)
    r = topk_retrieval_accuracy(a, b, ks=(1, 10))
    ok &= _check("unrelated embeddings sit at chance", r["top1"] < 0.02,
                 f"top1 = {r['top1']:.4f}, chance = {1/500:.4f}")
    ok &= _check("median rank of random retrieval is ~N/2",
                 100 < r["median_rank"] < 400, f"got {r['median_rank']}")

    # 4. contract
    r = topk_retrieval_accuracy(z, z.clone(), ks=(1, 3, 7))
    ok &= _check("respects the `ks` argument",
                 set(r) == {"top1", "top3", "top7", "median_rank"}, f"got keys {sorted(r)}")

    print("ALL TESTS PASSED\n" if ok else "SOME TESTS FAILED — fix the function above.\n")
    return ok


TEST2_OK = run_selftest(test_retrieval, "topk_retrieval_accuracy")

## 2. Training and evaluation harness

*(Given.)*

`train_alignment` trains the two heads with whatever loss you hand it and returns the test
embeddings. `evaluate` builds a fixed random gallery of 2000 test spots and calls *your*
retrieval function on it.

Two things to notice in the given code:

* **The gallery is 2000 spots**, so random guessing gives a top-1 accuracy of 1/2000 =
  0.05%. Keep that number in mind — it is the bar every result below has to clear.
* `_validate_loss_fn` runs your loss on a two-element dummy batch before training starts.
  This is a guard rail: it fails loudly in one second rather than after five minutes of
  training on a broken loss.

In [ ]:
GALLERY_SIZE = 2000
CHANCE_TOP1 = 1.0 / GALLERY_SIZE


def _validate_loss_fn(loss_fn, batch_size):
    """Fail fast and clearly if the loss function is not usable."""
    g = torch.Generator().manual_seed(0)
    a = F.normalize(torch.randn(batch_size, D_EMBED, generator=g), dim=-1).requires_grad_(True)
    b = F.normalize(torch.randn(batch_size, D_EMBED, generator=g), dim=-1)
    try:
        out = loss_fn(a, b)
    except NotImplementedError:
        raise RuntimeError(
            "The loss function has not been implemented yet — fill in the TODO above "
            "and re-run its self-test before starting the experiments."
        ) from None
    if not torch.is_tensor(out) or out.dim() != 0:
        raise RuntimeError(f"The loss must return a 0-dim tensor, got {out!r}.")
    if not torch.isfinite(out):
        raise RuntimeError("The loss returned NaN or inf on a dummy batch.")
    if out.grad_fn is None:
        raise RuntimeError(
            "The loss has no gradient. Did you detach something, or convert to numpy?"
        )


def train_alignment(loss_fn, epochs=15, batch_size=512, lr=1e-3, seed=0, verbose=False):
    """Train the two projection heads. Returns (z_img_test, z_expr_test)."""
    _validate_loss_fn(loss_fn, batch_size)
    set_seed(seed)

    f_img = ProjectionHead(D_IMG).to(DEVICE)
    f_expr = ProjectionHead(D_EXPR).to(DEVICE)
    opt = torch.optim.Adam(list(f_img.parameters()) + list(f_expr.parameters()), lr=lr)

    A, B = X_tr.to(DEVICE), E_tr.to(DEVICE)
    n = len(A)
    gen = torch.Generator().manual_seed(seed)

    for ep in range(epochs):
        perm = torch.randperm(n, generator=gen).to(DEVICE)
        running = 0.0
        nb = 0
        for i in range(0, n - batch_size + 1, batch_size):
            idx = perm[i:i + batch_size]
            loss = loss_fn(f_img(A[idx]), f_expr(B[idx]))
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += float(loss)
            nb += 1
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            print(f"    epoch {ep:2d}  loss {running / max(nb,1):.4f}")

    f_img.eval(); f_expr.eval()
    with torch.no_grad():
        return f_img(X_te.to(DEVICE)).cpu(), f_expr(E_te.to(DEVICE)).cpu()


def evaluate(z_img, z_expr, seed=0):
    """Retrieval on a fixed random gallery of GALLERY_SIZE test spots."""
    rng = np.random.default_rng(seed)
    sel = rng.choice(len(z_img), size=min(GALLERY_SIZE, len(z_img)), replace=False)
    q, g = z_img[sel], z_expr[sel]
    res = topk_retrieval_accuracy(q, g, ks=(1, 5, 10))
    # embedding spread: a healthy space uses many directions, a collapsed one does not
    res["embed_std"] = float(z_img.std(dim=0).mean())
    res["gallery"] = len(sel)
    return res


def show(tag, res):
    print(f"{tag:<26s} top1 {res['top1']:.3f} | top5 {res['top5']:.3f} | "
          f"top10 {res['top10']:.3f} | median rank {res['median_rank']:.0f} "
          f"/ {res['gallery']}")


print(f"gallery size {GALLERY_SIZE} -> random-guessing top-1 accuracy = {CHANCE_TOP1:.4%}")

## Experiment A — does contrastive alignment work at all?

One run with sensible defaults ($\tau = 0.07$, batch 512, 15 epochs). Compare the result
against the chance level printed above.

In [ ]:
assert TEST1_OK and TEST2_OK, "Fix TODO 1 and TODO 2 (and re-run their self-tests) first."

t0 = time.time()
z_img, z_expr = train_alignment(lambda a, b: info_nce_loss(a, b, 0.07), verbose=True)
res_main = evaluate(z_img, z_expr)
print(f"\ntrained in {time.time() - t0:.1f}s")
show("InfoNCE (tau=0.07)", res_main)
print(f"{'':26s} that is {res_main['top1'] / CHANCE_TOP1:.0f}x better than chance")

### What the shared space looks like

Two views of the same result. On the left, the similarity matrix for 40 test spots: if
alignment worked, the diagonal (true pairs) should be visibly brighter than everything
else. On the right, the distribution of true-pair similarities against random-pair
similarities — the gap between those two distributions *is* what the contrastive loss was
maximising.

In [ ]:
n_show = 40
sim = (z_img[:n_show] @ z_expr[:n_show].T).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
im = ax1.imshow(sim, cmap="viridis")
ax1.set_title("similarity matrix, 40 test spots", loc="left")
ax1.set_xlabel("expression spot"); ax1.set_ylabel("H&E patch")
fig.colorbar(im, ax=ax1, fraction=0.046, label="cosine similarity")

full = (z_img @ z_expr.T)
pos = full.diagonal().numpy()
neg = full[~torch.eye(len(full), dtype=bool)].numpy()
bins = np.linspace(-1, 1, 60)
ax2.hist(neg, bins=bins, density=True, color=C_MUTED, alpha=0.55, label="random pairs")
ax2.hist(pos, bins=bins, density=True, color=C_BLUE, alpha=0.85, label="true pairs")
ax2.set_title("do true pairs score higher?", loc="left")
ax2.set_xlabel("cosine similarity"); ax2.set_ylabel("density")
ax2.legend(frameon=False); ax2.grid(axis="y", alpha=0.6); ax2.set_axisbelow(True)
plt.tight_layout(); plt.show()

### Qualitative check — what does it actually retrieve?

For four query patches, we show the patch whose expression profile the model ranked
highest, second and third. The model never sees images at retrieval time — it is matching
*through* the expression space — so a retrieved patch looking similar to the query is
real evidence that morphology and expression have been tied together.

Green = the true partner was found. Grey = it was not.

In [ ]:
te_pos = {g: i for i, g in enumerate(IDX_te)}
thumb_rows = np.array([te_pos[g] for g in THUMB_IDX if g in te_pos])
thumb_imgs = np.array([THUMBS[j] for j, g in enumerate(THUMB_IDX) if g in te_pos])

qs = z_img[thumb_rows]
gs = z_expr[thumb_rows]
order = (qs @ gs.T).argsort(dim=1, descending=True)

set_seed(7)
picks = np.random.choice(len(thumb_rows), 4, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(7.5, 7.8))
for r, p in enumerate(picks):
    axes[r, 0].imshow(thumb_imgs[p]); axes[r, 0].set_ylabel("query", fontsize=8, color=C_MUTED)
    for c in range(3):
        hit = int(order[p, c]) == p
        axes[r, c + 1].imshow(thumb_imgs[int(order[p, c])])
        for sp in axes[r, c + 1].spines.values():
            sp.set_visible(True)
            sp.set_edgecolor(C_AQUA if hit else C_GRID)
            sp.set_linewidth(3.0 if hit else 1.0)
    for c in range(4):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
for c, t in enumerate(["query patch", "rank 1", "rank 2", "rank 3"]):
    axes[0, c].set_title(t, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## Experiment B — temperature

$\tau$ controls how sharply the loss punishes near-misses. A very small $\tau$ makes the
softmax extremely peaky, so the loss obsesses over the single hardest negative; a very
large $\tau$ flattens it until the loss barely distinguishes the true pair from anything
else. Everything else is held fixed, so the only thing changing here is $\tau$.

*(Given — this cell just calls your functions in a loop.)*

In [ ]:
TEMPERATURES = [0.005, 0.02, 0.07, 0.2, 1.0]
rows_temp = []
for tau in TEMPERATURES:
    r = evaluate(*train_alignment(lambda a, b, t=tau: info_nce_loss(a, b, t)))
    r["temperature"] = tau
    rows_temp.append(r)
    show(f"tau = {tau}", r)

df_temp = pd.DataFrame(rows_temp)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(df_temp.temperature, df_temp.top1, "-o", color=C_BLUE, lw=2, ms=7, label="top-1")
ax.plot(df_temp.temperature, df_temp.top10, "-o", color=C_ORANGE, lw=2, ms=7, label="top-10")
ax.axhline(CHANCE_TOP1, color=C_MUTED, ls="--", lw=1.2)
ax.text(0.006, CHANCE_TOP1 * 1.4, "chance", color=C_MUTED, fontsize=8)
best = df_temp.loc[df_temp.top1.idxmax()]
ax.annotate(f"best: tau={best.temperature}", (best.temperature, best.top1),
            textcoords="offset points", xytext=(6, 10), fontsize=9, color=C_INK)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("temperature (tau)"); ax.set_ylabel("retrieval accuracy")
ax.set_title("Temperature controls how hard the negatives push", loc="left")
ax.legend(frameon=False); ax.grid(alpha=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## Experiment C — how many negatives do you need?

InfoNCE learns by contrast, so the number of candidates each positive competes against
should matter. Naively you would test this by changing the batch size — but that would
also change the number of gradient steps and the effective learning rate, and you would
not know which of the three caused the difference.

Instead we hold the batch at 512 and **split it into independent chunks**. Chunk size 16
means every positive competes against 15 negatives; chunk size 512 means 511. The number
of optimisation steps, the samples seen and the learning rate are *identical* across all
settings, so the only variable is the number of negatives.

*(Given — note that it calls your `info_nce_loss` once per chunk.)*

In [ ]:
def chunked_info_nce(z_a, z_b, temperature, chunk):
    """Average your InfoNCE loss over independent chunks of the batch."""
    chunk = min(chunk, len(z_a))          # fall back to one chunk on a short batch
    losses = [
        info_nce_loss(z_a[i:i + chunk], z_b[i:i + chunk], temperature)
        for i in range(0, len(z_a) - chunk + 1, chunk)
    ]
    return torch.stack(losses).mean()


CHUNKS = [4, 16, 64, 256]
rows_neg = []
for m in CHUNKS:
    r = evaluate(*train_alignment(lambda a, b, m=m: chunked_info_nce(a, b, 0.07, m)))
    r["n_negatives"] = m - 1
    rows_neg.append(r)
    show(f"{m - 1} negatives", r)

df_neg = pd.DataFrame(rows_neg)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
x = np.arange(len(df_neg))
ax.bar(x - 0.19, df_neg.top1, 0.36, color=C_BLUE, label="top-1")
ax.bar(x + 0.19, df_neg.top10, 0.36, color=C_ORANGE, label="top-10")
for xi, (a, b) in enumerate(zip(df_neg.top1, df_neg.top10)):
    ax.text(xi - 0.19, a + 0.008, f"{a:.2f}", ha="center", fontsize=8, color=C_INK)
    ax.text(xi + 0.19, b + 0.008, f"{b:.2f}", ha="center", fontsize=8, color=C_INK)
ax.set_xticks(x); ax.set_xticklabels(df_neg.n_negatives)
ax.set_xlabel("negatives per positive"); ax.set_ylabel("retrieval accuracy")
ax.set_title("More negatives help — with diminishing returns", loc="left")
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## TODO 3 — what happens without negatives?

The lecture made a claim: contrastive learning needs negatives, and an objective that only
pulls true pairs *together* — with nothing pushing anything apart — has a trivial solution.
The model can map **every** input to the same vector, score a perfect loss, and learn
nothing. That is **representation collapse**, and it is the reason JEPA-style methods need
stop-gradients, momentum encoders and asymmetric architectures.

Let us verify the claim rather than believe it. Implement the naive objective: maximise
the cosine similarity of true pairs, and do nothing else.

$$\mathcal{L}_{\text{cos}} = -\frac{1}{N}\sum_{i=1}^{N} \mathrm{sim}(z_A^i, z_B^i)$$

In [ ]:
def cosine_alignment_loss(z_a: torch.Tensor, z_b: torch.Tensor) -> torch.Tensor:
    """Negative mean cosine similarity of the TRUE pairs only. No negatives involved.

    z_a, z_b : (B, D) L2-normalised tensors; row i of z_b is the true partner of row i
               of z_a. Returns a scalar tensor.
    """
    ### YOUR CODE HERE ###
    # Only the diagonal matters here: for each i, the similarity between z_a[i] and z_b[i].
    # Because both are L2-normalised, that is just their element-wise product summed over
    # the feature dimension. Average over the batch and negate (we minimise the loss but
    # want to maximise similarity).
    raise NotImplementedError("Implement cosine_alignment_loss")


def test_cosine_loss():
    print("Testing cosine_alignment_loss ...")
    ok = True
    v = F.normalize(torch.randn(16, 32, generator=torch.Generator().manual_seed(4)), dim=-1)
    ok &= _check("identical inputs give -1.0",
                 abs(float(cosine_alignment_loss(v, v.clone())) + 1.0) < 1e-5,
                 f"got {float(cosine_alignment_loss(v, v.clone())):.5f}")
    ok &= _check("opposite inputs give +1.0",
                 abs(float(cosine_alignment_loss(v, -v)) - 1.0) < 1e-5)
    e1 = torch.zeros(4, 8); e1[:, 0] = 1.0
    e2 = torch.zeros(4, 8); e2[:, 1] = 1.0
    ok &= _check("orthogonal inputs give 0.0",
                 abs(float(cosine_alignment_loss(e1, e2))) < 1e-6)
    ok &= _check("only the diagonal is used (shuffling the batch changes the value)",
                 abs(float(cosine_alignment_loss(v, v.flip(0))) + 1.0) > 1e-3)
    print("ALL TESTS PASSED\n" if ok else "SOME TESTS FAILED — fix the function above.\n")
    return ok


TEST3_OK = run_selftest(test_cosine_loss, "cosine_alignment_loss")

### The collapse experiment

Train with the negative-free loss and look at two numbers:

* **retrieval accuracy** — should fall back towards chance, and
* **embedding spread** (`embed_std`, the mean per-dimension standard deviation of the test
  embeddings) — the fingerprint of collapse. If the model has mapped every patch to nearly
  the same point, this number is near zero.

In [ ]:
assert TEST3_OK, "Fix TODO 3 first."

z_i_c, z_e_c = train_alignment(cosine_alignment_loss)
res_collapse = evaluate(z_i_c, z_e_c)
show("cosine only (no negatives)", res_collapse)
print()
print(f"  embedding spread, InfoNCE      : {res_main['embed_std']:.4f}")
print(f"  embedding spread, cosine only  : {res_collapse['embed_std']:.4f}")
print(f"  ratio                          : {res_main['embed_std'] / max(res_collapse['embed_std'], 1e-9):.1f}x")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
labels = ["InfoNCE\n(with negatives)", "cosine only\n(no negatives)"]
cols = [C_BLUE, C_ORANGE]

vals = [res_main["top1"], res_collapse["top1"]]
ax1.bar(labels, vals, color=cols, width=0.55)
ax1.axhline(CHANCE_TOP1, color=C_MUTED, ls="--", lw=1.2)
ax1.text(1.42, CHANCE_TOP1 * 1.5, "chance", color=C_MUTED, fontsize=8, ha="right")
for i, v in enumerate(vals):
    ax1.text(i, v * 1.15, f"{v:.3f}", ha="center", fontsize=9, color=C_INK)
ax1.set_yscale("log"); ax1.set_ylabel("top-1 retrieval accuracy")
ax1.set_title("Retrieval", loc="left")
ax1.grid(axis="y", alpha=0.6); ax1.set_axisbelow(True)

vals2 = [res_main["embed_std"], res_collapse["embed_std"]]
ax2.bar(labels, vals2, color=cols, width=0.55)
for i, v in enumerate(vals2):
    ax2.text(i, v, f"  {v:.4f}", ha="center", va="bottom", fontsize=9, color=C_INK)
ax2.set_ylabel("mean per-dimension std")
ax2.set_title("Embedding spread (collapse fingerprint)", loc="left")
ax2.grid(axis="y", alpha=0.6); ax2.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 3. Summary — write out your deliverables

*(Given.)* This cell collects every result into one table and one figure, and writes both
to disk. **Submit `part1_results.csv` and `part1_figure.png` together with the notebook.**

In [ ]:
def summarise_part1():
    rows = [dict(experiment="baseline", setting="tau=0.07, 511 negatives", **res_main)]
    for r in rows_temp:
        rows.append(dict(experiment="temperature", setting=f"tau={r['temperature']}", **r))
    for r in rows_neg:
        rows.append(dict(experiment="negatives", setting=f"{r['n_negatives']} negatives", **r))
    rows.append(dict(experiment="no-negatives", setting="cosine only", **res_collapse))

    keep = ["experiment", "setting", "top1", "top5", "top10", "median_rank",
            "embed_std", "gallery"]
    df = pd.DataFrame(rows)[keep]
    df["chance_top1"] = CHANCE_TOP1
    df.to_csv("part1_results.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9))

    axes[0].plot(df_temp.temperature, df_temp.top1, "-o", color=C_BLUE, lw=2, ms=6)
    axes[0].set_xscale("log"); axes[0].set_xlabel("temperature")
    axes[0].set_ylabel("top-1 accuracy"); axes[0].set_title("B · temperature", loc="left")

    axes[1].bar(np.arange(len(df_neg)), df_neg.top1, 0.6, color=C_BLUE)
    axes[1].set_xticks(np.arange(len(df_neg)))
    axes[1].set_xticklabels(df_neg.n_negatives)
    axes[1].set_xlabel("negatives per positive")
    axes[1].set_ylabel("top-1 accuracy")
    axes[1].set_title("C · number of negatives", loc="left")

    axes[2].bar(["InfoNCE", "cosine only"], [res_main["top1"], res_collapse["top1"]],
                0.55, color=[C_BLUE, C_ORANGE])
    axes[2].set_yscale("log")
    axes[2].set_ylabel("top-1 accuracy (log scale)")
    axes[2].set_title("D · negatives are load-bearing", loc="left")

    for ax in axes:
        ax.axhline(CHANCE_TOP1, color=C_MUTED, ls="--", lw=1.1)
        ax.grid(axis="y", alpha=0.6); ax.set_axisbelow(True)
    fig.suptitle("Part 1 · contrastive alignment of H&E patches and ST spots  "
                 f"(gallery {GALLERY_SIZE}, dashed line = chance)", x=0.01, ha="left")
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.savefig("part1_figure.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nwrote part1_results.csv and part1_figure.png")
    return df


results_part1 = summarise_part1()
results_part1.round(4)

## 4. Questions

Answer briefly — two or three sentences each — in the cells below.

---

**Q1.** Your best model retrieves the correct expression profile out of 2000 candidates far
more often than chance, but top-1 accuracy is still well below 100%. Give one reason
rooted in the *biology* (not in the optimisation) why perfect retrieval is not achievable
here, and relate it to the information-theoretic picture from the lecture.

*Your answer:*


**Q2.** In Experiment B, both a very small and a very large temperature hurt, for
different reasons. Explain each side of the curve.

*Your answer:*


**Q3.** In the lecture we warned that in WSI/ST data, a "negative" spot may be
biologically almost identical to the positive one. Look at your Experiment C results: does
adding more negatives keep helping indefinitely? Explain what you observe, and say what
you would expect to happen if you could keep growing the number of negatives towards the
whole dataset.

*Your answer:*


**Q4.** The cosine-only model reaches a *lower loss value* than the InfoNCE model, yet is
useless for retrieval. What does this tell you about using the training loss to judge a
representation-learning method? Name the diagnostic in this notebook that exposed the
problem.

*Your answer:*


**Q5.** The split in this notebook is by slide. Suppose you had split the spots at random
instead. Would your retrieval accuracy go up or down, and would the resulting number mean
what you want it to mean? (One or two sentences.)

*Your answer:*
